In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3

In [ ]:
stocks = {
    # Banking
    "HDFCBANK.NS": "Banking",
    "ICICIBANK.NS": "Banking",
    "SBIN.NS": "Banking",
    # IT
    "TCS.NS": "IT",
    "INFY.NS": "IT",
    "WIPRO.NS": "IT",
    # FMCG
    "INDUNHILVR.NS": "FMCG",
    "ITC.NS": "FMCG",
    "NESTLEIND.NS": "FMCG",
    # Auto
    "MARUTI.NS": "Auto",
    "EICHERMOT .NS": "Auto",
    "M&M.NS": "Auto",
    # Pharma
    "SUNPHARMA.NS": "Pharma",
    "CIPLA.NS": "Pharma",
    "DRREDDY.NS": "Pharma",
}
benchmark = "^NSEI"  # Nifty 50 index

In [3]:
tickers = list(stocks.keys()) + [benchmark]

price_data = {}

for ticker in tickers:
    print(f"Downloading {ticker}...")
    
    df = yf.download(
        ticker,
        start="2021-01-01",
        end="2026-01-01",
        auto_adjust=False,
        progress=False
    )
    
    # Check if download failed
    if df.empty:
        print(f"⚠️ No data found for {ticker} - skipping")
        continue
    
    # Get Close price
    close = df["Close"]
    
    # Convert to Series if needed
    if isinstance(close, pd.DataFrame):
        close = close.iloc[:, 0]
    
    price_data[ticker] = close

# Combine everything
prices = pd.concat(price_data, axis=1)

print("\nFinal shape:", prices.shape)
prices.head()


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: .NS"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: EICHERMOT"}}}
$EICHERMOT: possibly delisted; no timezone found
$.NS: possibly delisted; no timezone found

2 Failed downloads:
['EICHERMOT', '.NS']: possibly delisted; no timezone found


⚠️ No data found for EICHERMOT .NS - skipping

Final shape: (1236, 15)


,HDFCBANK.NS,ICICIBANK.NS,SBIN.NS,TCS.NS,INFY.NS,WIPRO.NS,HINDUNILVR.NS,ITC.NS,NESTLEIND.NS,MARUTI.NS,M&M.NS,SUNPHARMA.NS,CIPLA.NS,DRREDDY.NS,^NSEI
Date,,,,,,,,,,,,,,,
2021-01-01,712.525024,527.500000,279.399994,2928.250000,1260.449951,194.050003,2387.550049,205.919418,922.534973,7691.299805,732.450012,596.250000,826.599976,1048.270020,14018.500000
2021-01-04,708.000000,531.700012,281.049988,3039.449951,1288.250000,198.199997,2426.500000,205.486115,918.897522,7702.299805,749.099976,604.400024,832.250000,1054.449951,14132.900391
2021-01-05,713.349976,537.250000,281.750000,3093.000000,1293.800049,203.149994,2450.550049,203.608429,927.912476,7655.450195,740.099976,603.450012,827.250000,1057.380005,14199.500000
2021-01-06,710.275024,546.700012,285.049988,3051.500000,1282.099976,203.199997,2417.300049,197.782791,925.762512,7628.600098,736.099976,605.299988,824.799988,1057.660034,14146.250000
2021-01-07,708.125000,541.099976,287.700012,3032.800049,1262.150024,203.375000,2368.850098,195.279205,906.364990,7566.049805,744.400024,601.900024,826.549988,1054.180054,14137.349609


In [4]:
price_data

{'HDFCBANK.NS': Date
 2021-01-01    712.525024
 2021-01-04    708.000000
 2021-01-05    713.349976
 2021-01-06    710.275024
 2021-01-07    708.125000
                  ...    
 2025-12-24    997.200012
 2025-12-26    992.099976
 2025-12-29    991.700012
 2025-12-30    990.900024
 2025-12-31    991.200012
 Name: HDFCBANK.NS, Length: 1236, dtype: float64,
 'ICICIBANK.NS': Date
 2021-01-01     527.500000
 2021-01-04     531.700012
 2021-01-05     537.250000
 2021-01-06     546.700012
 2021-01-07     541.099976
                  ...     
 2025-12-24    1359.800049
 2025-12-26    1350.400024
 2025-12-29    1343.300049
 2025-12-30    1342.500000
 2025-12-31    1342.900024
 Name: ICICIBANK.NS, Length: 1236, dtype: float64,
 'SBIN.NS': Date
 2021-01-01    279.399994
 2021-01-04    281.049988
 2021-01-05    281.750000
 2021-01-06    285.049988
 2021-01-07    287.700012
                  ...    
 2025-12-24    968.950012
 2025-12-26    966.299988
 2025-12-29    965.049988
 2025-12-30    973.450

## Cleaning the data

In [5]:
prices.isna().sum()

HDFCBANK.NS      0
ICICIBANK.NS     0
SBIN.NS          0
TCS.NS           0
INFY.NS          0
WIPRO.NS         0
HINDUNILVR.NS    0
ITC.NS           0
NESTLEIND.NS     0
MARUTI.NS        0
M&M.NS           0
SUNPHARMA.NS     0
CIPLA.NS         0
DRREDDY.NS       0
^NSEI            0
dtype: int64

In [6]:
threshold = 0.05*len(prices)
prices_clean = prices.dropna(axis=1,thresh=len(prices) - threshold)

prices_clean = prices_clean.ffill()
print(prices_clean.shape)

(1236, 15)


### Saving it to excel

In [7]:
prices_clean.to_csv(r'C:\Users\saisu\Desktop\Project\prices_clean.csv')

### Connecting to a database

In [13]:
conn = sqlite3.connect("C:\\Users\\saisu\\Desktop\\Project\\portfolio.db")

In [17]:
price_data = pd.DataFrame(price_data)

In [18]:
price_data.to_sql('daily_prices',conn,if_exists = 'replace')

1236

### Creating a table to see which sector each stock belongs to

In [19]:
metadata = pd.DataFrame(
    [{"Ticker": t, "Sector": s} for t, s in stocks.items()]
)

In [21]:
metadata.to_sql('stock_meta',conn,if_exists='replace',index=False)

conn.commit()

### Viewing all the closing prices of FMCG stocks

In [25]:
# Check existing column names in the daily_prices table
pd.read_sql("PRAGMA table_info(daily_prices)", conn)

,cid,name,type,notnull,dflt_value,pk
0,0,Date,TIMESTAMP,0,None,0
1,1,HDFCBANK.NS,REAL,0,None,0
2,2,ICICIBANK.NS,REAL,0,None,0
3,3,SBIN.NS,REAL,0,None,0
4,4,TCS.NS,REAL,0,None,0
5,5,INFY.NS,REAL,0,None,0
6,6,WIPRO.NS,REAL,0,None,0
7,7,HINDUNILVR.NS,REAL,0,None,0
8,8,ITC.NS,REAL,0,None,0
9,9,NESTLEIND.NS,REAL,0,None,0


In [27]:
query = """
SELECT Date, "HINDUNILVR.NS", "ITC.NS", "NESTLEIND.NS"
FROM daily_prices
ORDER BY Date
"""

fmcg_prices = pd.read_sql(query, conn)
fmcg_prices.head()

,Date,HINDUNILVR.NS,ITC.NS,NESTLEIND.NS
0,2021-01-01 00:00:00,2387.550049,205.919418,922.534973
1,2021-01-04 00:00:00,2426.500000,205.486115,918.897522
2,2021-01-05 00:00:00,2450.550049,203.608429,927.912476
3,2021-01-06 00:00:00,2417.300049,197.782791,925.762512
4,2021-01-07 00:00:00,2368.850098,195.279205,906.364990


### Viewing all the closing prices of IT stocks

In [36]:
query = """
SELECT Date, "TCS.NS", "INFY.NS", "WIPRO.NS"
FROM daily_prices
ORDER BY Date
"""

it_prices = pd.read_sql(query, conn)
it_prices.head()

,Date,TCS.NS,INFY.NS,WIPRO.NS
0,2021-01-01 00:00:00,2928.250000,1260.449951,194.050003
1,2021-01-04 00:00:00,3039.449951,1288.250000,198.199997
2,2021-01-05 00:00:00,3093.000000,1293.800049,203.149994
3,2021-01-06 00:00:00,3051.500000,1282.099976,203.199997
4,2021-01-07 00:00:00,3032.800049,1262.150024,203.375000


### Recent 10 day trading IT stocks


In [39]:
query = """
SELECT Date, "TCS.NS", "INFY.NS", "WIPRO.NS"
FROM daily_prices
ORDER BY Date DESC
LIMIT 10
"""

IT_prices = pd.read_sql(query, conn)
IT_prices

,Date,TCS.NS,INFY.NS,WIPRO.NS
0,2025-12-31 00:00:00,3206.199951,1615.400024,263.279999
1,2025-12-30 00:00:00,3246.800049,1621.599976,263.649994
2,2025-12-29 00:00:00,3251.500000,1644.699951,264.239990
3,2025-12-26 00:00:00,3280.000000,1656.099976,266.299988
4,2025-12-24 00:00:00,3319.000000,1663.400024,268.059998
5,2025-12-23 00:00:00,3310.000000,1668.300049,271.399994
6,2025-12-22 00:00:00,3324.899902,1689.599976,272.670013
7,2025-12-19 00:00:00,3282.000000,1638.699951,264.450012
8,2025-12-18 00:00:00,3280.800049,1626.800049,263.850006
9,2025-12-17 00:00:00,3217.800049,1602.000000,261.140015


In [38]:
it_prices.head()

,Date,TCS.NS,INFY.NS,WIPRO.NS
0,2021-01-01 00:00:00,2928.250000,1260.449951,194.050003
1,2021-01-04 00:00:00,3039.449951,1288.250000,198.199997
2,2021-01-05 00:00:00,3093.000000,1293.800049,203.149994
3,2021-01-06 00:00:00,3051.500000,1282.099976,203.199997
4,2021-01-07 00:00:00,3032.800049,1262.150024,203.375000
